# 153 — Layer Transition Analysis

How does each layer change the residual stream? Some layers make large changes,
others act almost as identity. This module measures transition magnitude,
smoothness, and which layers are most critical for prediction.

## Why This Matters

Layer transition characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_transition_analysis import (
    layer_transition_magnitude,
    component_transition_contribution,
    transition_smoothness,
    identity_layers,
    critical_transitions,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)

key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.arange(15)

## Transition Magnitude

How much does each layer change the residual stream? Large relative changes
mean the layer is doing significant work.

In [ ]:
result = layer_transition_magnitude(model, tokens)

print(f"Max change layer: {result['max_change_layer']}")
print(f"Mean relative change: {result['mean_relative_change']:.4f}\n")

for p in result['per_layer']:
    bar = '#' * int(p['relative_change'] * 30)
    print(f"Layer {p['layer']}: rel_change={p['relative_change']:.4f}  "
          f"dir_preserve={p['direction_preservation']:.4f}  {bar}")

## Component Contribution Breakdown

Is the change driven by attention or MLP? Do they cooperate or oppose?

In [ ]:
result = component_transition_contribution(model, tokens)

print(f"Attn-dominant layers: {result['attn_dominant_layers']}")
print(f"MLP-dominant layers: {result['mlp_dominant_layers']}\n")

for p in result['per_layer']:
    coop = 'COOP' if p['cooperative'] else 'OPP '
    print(f"Layer {p['layer']}: attn={p['attn_fraction']:.2f} mlp={p['mlp_fraction']:.2f}  "
          f"cos={p['attn_mlp_cosine']:+.3f} [{coop}]")

## Transition Smoothness

Do adjacent layers make similar or different kinds of changes?

In [ ]:
result = transition_smoothness(model, tokens)

print(f"Smoothness score: {result['smoothness_score']:.2f}")
print(f"Smooth: {result['n_smooth']}, Abrupt: {result['n_abrupt']}\n")

for t in result['transitions']:
    tag = 'SMOOTH' if t['smooth'] else 'ABRUPT'
    print(f"L{t['from_layer']}->L{t['to_layer']}: dir_sim={t['direction_similarity']:+.3f}  "
          f"mag_ratio={t['magnitude_ratio']:.3f}  [{tag}]")

## Identity Layer Detection

Near-identity layers barely change the representation — potentially redundant.

In [ ]:
result = identity_layers(model, tokens, threshold=0.5)

print(f"Identity layers (threshold={result['threshold']}): {result['identity_layers']}\n")
for p in result['per_layer']:
    tag = ' <-- IDENTITY' if p['is_identity'] else ''
    print(f"Layer {p['layer']}: mean_change={p['mean_relative_change']:.4f}{tag}")

## Critical Transitions

Which layers change the prediction logit the most? Breaks down into attention
and MLP contributions to the logit.

In [ ]:
result = critical_transitions(model, tokens)

print(f"Target token: {result['target_token']}")
print(f"Most critical layer: {result['most_critical_layer']}\n")

for p in result['per_layer']:
    print(f"Layer {p['layer']}: logit {p['logit_before']:+.4f} -> {p['logit_after']:+.4f}  "
          f"delta={p['logit_delta']:+.4f} (attn={p['attn_logit_delta']:+.4f} mlp={p['mlp_logit_delta']:+.4f})")